In [0]:
from pyspark.sql.functions import col, broadcast, rand, concat, lit, array, explode, md5, concat_ws

###1. Read Clean Data from Silver ADLS Container

In [0]:
silver_path = "abfss://silver@migrationecom.dfs.core.windows.net/cleansed_orders"
df_silver = spark.read.format("delta")\
                .load(silver_path)

In [0]:
df_silver.display()

###2. ENTERPRISE JOINS (Broadcast & Salted)

In [0]:
region_data = [("U554", "North_Region"), ("U555", "South_Region")]
df_region = spark.createDataFrame(region_data, ['user_id', 'region_name'])

In [0]:
df_region.display()

#### Broadcast Join

In [0]:
df_broadcast_joined = df_silver.join(broadcast(df_region), ["user_id"], "left")

In [0]:
df_broadcast_joined.display()

In [0]:
sla_data = [("DELIVERED", 24), ("CANCELLED", 0), ("SHIPPED", 48), ("PENDING", 72), ("PROCESSING", 12)]
df_sla = spark.createDataFrame(sla_data, ['status_key', 'sla_hour'])

In [0]:
df_sla.display()

#### Salting

Adding salted key in skewed table

In [0]:
df_orders_salted = df_broadcast_joined.withColumn("salt", (rand()*5).cast('int'))\
                                    .withColumn("salted_key", concat(col('order_status'), lit('_'), col('salt')))

In [0]:
df_orders_salted.display()

Exploding the other table with salted keys

In [0]:
salt_array = array([lit(i) for i in range(5)])

df_sla_exploded = df_sla.withColumn("salt_array", salt_array)\
                        .withColumn("salt_exploded", explode(col("salt_array")))\
                        .withColumn("salted_key", concat(col("status_key"), lit("_"), col("salt_exploded")))

In [0]:
df_sla_exploded.display()

In [0]:
df_gold = df_orders_salted.join(df_sla_exploded, ["salted_key"], "left")

In [0]:
df_gold.display()

Dropping unnecessary columns

In [0]:
df_gold_clean = df_gold.drop("salt", "salt_array", "salt_exploded", "salted_key", "status_key")

In [0]:
df_gold_clean.display()

### 3. HASHING & SNOWFLAKE TARGET LOAD

#### Hashing

In [0]:
df_final = df_gold_clean.withColumn("md5", md5(concat_ws('||', col("order_id"), col("user_id"), col("order_status"), col("order_date"))))

In [0]:
df_final.display()

Loading data to Snowflkake Target table

In [0]:
sfUrl = "DAPPSDF-RV26711.snowflakecomputing.com"
sfuser = dbutils.secrets.get(scope="snowflake_scope", key="username")
sfPassword = dbutils.secrets.get(scope="snowflake_scope", key="password")

sfOptions = {
    "sfUrl" : sfUrl,
    "sfUser" : sfuser,
    "sfPassword" : sfPassword,
    "sfDatabase" : "ECOM_DB",
    "sfSchema" : "MIGRATION",
    "sfWarehouse" : "COMPUTE_WH"
}

df_final.write.format("snowflake")\
        .options(**sfOptions)\
        .option("dbtable", "TARGET_ORDERS")\
        .mode("overwrite")\
        .save()